In [1]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,Unnamed: 0,id,gen,judet,NT,MEV,MATE,MGIM,status_admitere
0,0,407,F,BT,4.91,8.82,6.23,6.85,0
1,1,444,F,BC,9.50,7.31,8.97,8.36,1
2,2,117,F,BH,6.23,8.63,5.40,8.54,0
3,3,30,M,SM,1.35,9.49,9.06,5.59,0
4,4,415,M,OT,1.82,6.71,8.67,8.34,0


In [2]:
df_test.isna().sum()

Unnamed: 0    0
id            0
gen           0
judet         0
NT            0
MEV           0
MATE          0
MGIM          0
dtype: int64

In [3]:
df_train.columns

Index(['Unnamed: 0', 'id', 'gen', 'judet', 'NT', 'MEV', 'MATE', 'MGIM',
       'status_admitere'],
      dtype='object')

In [4]:
features = ['gen', 'judet', 'NT', 'MEV', 'MATE', 'MGIM']
target = 'status_admitere'

X = df_train[features]
y = df_train[target]

X_final = df_test[features]

from sklearn.preprocessing import LabelEncoder

enc = LabelEncoder()

X.judet = enc.fit_transform(X.judet)
X_final.judet = enc.transform(X_final.judet)

X.gen = X.gen.map({"M": 0, "F": 1})
X_final.gen = X_final.gen.map({"M": 0, "F": 1})

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X = pd.DataFrame(scaler.fit_transform(X), columns=features)
X_final = pd.DataFrame(scaler.fit_transform(X_final), columns=features)

X.head()

C:\Users\Ilie-Goga\AppData\Local\Temp\ipykernel_17184\1481530172.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.judet = enc.fit_transform(X.judet)
C:\Users\Ilie-Goga\AppData\Local\Temp\ipykernel_17184\1481530172.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_final.judet = enc.transform(X_final.judet)
C:\Users\Ilie-Goga\AppData\Local\Temp\ipykernel_17184\1481530172.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col

,gen,judet,NT,MEV,MATE,MGIM
0,0.956501,-1.145317,-0.132210,0.626925,-0.947687,-0.524319
1,0.956501,-1.480440,1.710842,-0.662863,0.988034,0.545522
2,0.956501,-1.396659,0.397818,0.464634,-1.534054,0.673053
3,-1.045478,1.032988,-1.561678,1.199215,1.051616,-1.417034
4,-1.045478,0.697864,-1.372956,-1.175362,0.776094,0.531352


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_score

model = LinearSVC(random_state=42)

cross_val_score(model, X, y, cv=10, scoring='f1')

array([0.92307692, 0.92307692, 0.93333333, 0.93333333, 1.        ,
       0.93333333, 1.        , 1.        , 1.        , 1.        ])

In [6]:
model.fit(X, y)
y_final = model.predict(X_final)

In [7]:
submission = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.id,
    "answer": y_final
})

submission.head()

,subtaskID,datapointID,answer
0,3,289,1
1,3,312,1
2,3,423,1
3,3,348,0
4,3,415,0


In [8]:
subtask_1 = (df_test.NT - df_test.MEV).round(2)

df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.id,
    "answer": subtask_1
})

clasament = df_test[["id", "MEV"]]
clasament = clasament.sort_values(by="MEV", ignore_index=True, ascending=False)
subtask_2 = []
for _, row in df_test.iterrows():
    idx = row.id
    subtask_2.append(clasament[clasament.id==idx].index[0]+1)

df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.id,
    "answer": subtask_2
})

In [9]:
df_final = pd.concat([df_1, df_2, submission])

df_final

,subtaskID,datapointID,answer
0,1,289,3.28
1,1,312,-1.00
2,1,423,-0.69
3,1,348,-0.74
4,1,415,-4.89
...,...,...,...
175,3,341,0.00
176,3,376,0.00
177,3,284,0.00
178,3,362,0.00


In [10]:
df_final.to_csv("submission.csv", index=False)